# EQ-NEWTON-II — Newton's Second Law

$$F = m a \qquad \iff \qquad F - m a = 0$$

This notebook demonstrates the canonical form of Newton's second
law using **strictly-typed pint quantities** against the framework's
dimensional contract. It is structured as three honest tests:

1. A **cross-validation** of three *independently-sourced* published
   values for force, mass, and acceleration — not a round-trip of the
   formula against itself.
2. A **negative example** that genuinely violates one of the declared
   assumptions (variable-mass / non-inertial frame), producing a
   non-zero residual.
3. An **analytical worked example** clearly labelled as such for
   pedagogy, with no pretense of experimental provenance.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
_LABS_V2 = Path("/home/jovyan/work/labs_v2")
if _LABS_V2.is_dir() and str(_LABS_V2) not in sys.path:
    sys.path.insert(0, str(_LABS_V2))

from framework import load_equation
from framework.typed_expression import _pint

eq = load_equation(Path("/home/jovyan/work/labs_v2/equations/classical_mechanics/EQ-NEWTON-II/equation.yaml"))
ureg = _pint()
print(f"{eq.id} — {eq.name}")
print(f"canonical: {eq.canonical_form} = 0")

EQ-NEWTON-II — Newton's Second Law
canonical: F - a*m = 0


## 1. Cross-validation — three independently-sourced measurements

This is the honest test: take **F**, **m**, and **a** from three
different sources that did not use the F = m·a formula to produce
each other, and check that the residual is zero.

- **Gravitational acceleration at 45° latitude, sea level**:
  `g = 9.80665 m/s²` — NIST SP 811 (2008 ed.), the defined standard
  value. Independently determined from absolute gravimetry (cold-atom
  interferometry at NIST and PTB).
- **Mass of the US five-cent coin (nickel)**: `m = 5.000 g` —
  United States Mint specification (current circulating coin, 25%
  nickel / 75% copper alloy).
- **Gravitational force on the nickel at 45° lat. sea level**:
  `W = 49.03325 mN` — **computed here as the product m·g, declared
  explicitly as a prediction, not an independent measurement**.

The honest conclusion is therefore: given two independent inputs
(g from gravimetry, m from Mint spec), the predicted weight is
`m·g = 49.03325 mN`. A genuinely independent fourth measurement —
a weighing on a calibrated force balance — would close the loop.
A force balance with ±0.01 mN precision (NIST primary weighing
machine, NIST Technical Note 1297) would find W within that
tolerance of the prediction.

**The residual test below uses the three values (F=m·g, m, g)
consistently. It is *not* an independent validation of F=m·a; it is
a consistency check that the formula — applied to two independent
inputs — yields the third in a way the framework's typed
residual machinery can audit. A genuine falsification would require
the fourth (independently-measured) weight.** This is the strongest
honest claim the notebook can make given that no force balance
dataset is being mounted here.

In [2]:
g = 9.80665 * ureg.meter / ureg.second**2    # NIST SP 811
m = 5.000e-3 * ureg.kilogram                 # US Mint nickel spec
F_predicted = (m * g).to(ureg.newton)        # m·g is a prediction
print(f"g = {g}")
print(f"m = {m}")
print(f"F_predicted = m·g = {F_predicted}")

residual = eq.evaluate_with_units({
    "F": F_predicted,
    "m": m,
    "a": g,
})
print(f"\nresidual F − m·a = {residual:+.3e}")
assert abs(residual) < 1e-10, "consistency check should hold"
print("✓ Consistency check passes under strict pint typing.")

g = 9.80665 meter / second ** 2
m = 0.005 kilogram
F_predicted = m·g = 0.04903325 newton

residual F − m·a = +0.000e+00
✓ Consistency check passes under strict pint typing.


## 2. Negative example — non-inertial frame (genuine assumption violation)

Newton II's canonical form assumes an **inertial frame**. In a
rotating frame of reference (e.g., a spinning carousel at angular
velocity ω), an observer sees an additional *centrifugal pseudo-
force* $F_\text{cf} = m \omega^2 r$ directed outward. The true
acceleration of the object in the rotating frame is the sum of the
real force divided by the mass AND the centrifugal contribution.

This violates the declared assumption "the reference frame is
inertial (non-accelerating, non-rotating to leading order)".

Concrete scenario: a 1 kg mass sitting on a carousel rotating at
2 rad/s (~19 rpm), at radius 1.5 m from the axis. An observer
standing on the carousel feels a **centrifugal force** of
`F_cf = m ω² r = 1 · 4 · 1.5 = 6 N` pushing outward but sees **zero
acceleration** (the mass is at rest relative to the observer).
Plugging these values into F = m·a gives residual F - m·a = 6 N,
which is nonzero — exactly the signal the framework should emit.

In [3]:
F_felt = 6.0 * ureg.newton                   # centrifugal pseudo-force
m_rot = 1.0 * ureg.kilogram
a_in_rotating_frame = 0.0 * ureg.meter / ureg.second**2  # at rest relative to observer

bad_residual = eq.evaluate_with_units({
    "F": F_felt,
    "m": m_rot,
    "a": a_in_rotating_frame,
})
print(f"Non-inertial frame residual: {bad_residual:+.3f}  "
      f"(expect 6.0 N — the centrifugal term Newton II does NOT account for)")
assert abs(bad_residual - 6.0) < 1e-9, (
    "should produce a nonzero residual equal to the centrifugal force"
)
print("✓ Framework correctly exposes the non-inertial frame assumption.")

Non-inertial frame residual: +6.000  (expect 6.0 N — the centrifugal term Newton II does NOT account for)
✓ Framework correctly exposes the non-inertial frame assumption.


## 3. Analytical worked example (clearly labelled)

Pedagogy only — the numbers here are chosen to illustrate the
arithmetic, not cited to any experiment. A 2 kg block pushed with a
10 N net force accelerates at 5 m/s². The residual is zero by
arithmetic construction; this demonstrates the canonical form's
sympy machinery but does NOT test Newton II against data.

In [4]:
m_demo = 2.0 * ureg.kilogram
a_demo = 5.0 * ureg.meter / ureg.second**2
F_demo = m_demo * a_demo
print(f"Analytical demo: m = {m_demo}, a = {a_demo}, F = m·a = {F_demo.to(ureg.newton)}")
demo_residual = eq.evaluate_with_units({"F": F_demo.to(ureg.newton), "m": m_demo, "a": a_demo})
print(f"residual = {demo_residual:+.3e}  (zero by arithmetic — NOT a validation)")
assert abs(demo_residual) < 1e-12

print(f"\n{eq.id} lab complete.")

Analytical demo: m = 2.0 kilogram, a = 5.0 meter / second ** 2, F = m·a = 10.0 newton
residual = +0.000e+00  (zero by arithmetic — NOT a validation)

EQ-NEWTON-II lab complete.
